In [2]:
import pandas as pd
import numpy as np
import folium

from folium.plugins import HeatMap, MiniMap, Fullscreen, MeasureControl, BeautifyIcon
from pathlib import Path

BASE_DIR = Path("../DATA/PROCESS")
LOCAL_DIR = Path("../DATA/LOCAL")
OUTPUT_DIR = Path("../RESULT")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RAW_PATH = BASE_DIR / "save_dataset3.csv"
FINAL_CANDIDATE_PATH = BASE_DIR / "final_candidate_local_df.csv"
SELECTED_PATH = BASE_DIR / "facility_location_best_selected_j.csv"
LOCAL_PATH = LOCAL_DIR / "local.csv"

OUTPUT_PATH = OUTPUT_DIR / "final_hidden_spot_map_final.html"

LAT_COL = "위도"
LON_COL = "경도"


def read_csv_safely(path, usecols=None):
    try:
        return pd.read_csv(path, usecols=usecols)
    except UnicodeDecodeError:
        return pd.read_csv(path, encoding="cp949", usecols=usecols)


def clean_geo_df(df):
    temp = df.copy()

    temp[LAT_COL] = pd.to_numeric(temp[LAT_COL], errors="coerce")
    temp[LON_COL] = pd.to_numeric(temp[LON_COL], errors="coerce")

    temp = temp.dropna(subset=[LAT_COL, LON_COL])

    temp = temp[
        (temp[LAT_COL].between(37.3, 37.8)) &
        (temp[LON_COL].between(126.7, 127.3))
    ].copy()

    return temp


raw_usecols = [
    "target",
    "위도",
    "경도",
    "hour",
    "weekday",
    "자치구코드"
]

raw_df = read_csv_safely(RAW_PATH, usecols=raw_usecols)
candidate_df = read_csv_safely(FINAL_CANDIDATE_PATH)
selected_df = read_csv_safely(SELECTED_PATH)
local_df = read_csv_safely(LOCAL_PATH)

H_map = clean_geo_df(raw_df[raw_df["target"] == 1])
candidate_map = clean_geo_df(candidate_df)
selected_map = clean_geo_df(selected_df)
local_map = clean_geo_df(local_df)

selected_map = selected_map.sort_values("selection_order").reset_index(drop=True)

center_source = selected_map if len(selected_map) > 0 else candidate_map

m = folium.Map(
    location=[
        center_source[LAT_COL].mean(),
        center_source[LON_COL].mean()
    ],
    zoom_start=11,
    tiles="cartodbpositron",
    prefer_canvas=True
)

Fullscreen().add_to(m)
MiniMap(toggle_display=True).add_to(m)
MeasureControl(primary_length_unit="meters").add_to(m)

h_layer = folium.FeatureGroup(
    name=f"1단계: 혼잡 지역 H Heatmap ({len(H_map):,})",
    show=True
)

candidate_layer = folium.FeatureGroup(
    name=f"2단계: 전체 후보 J Heatmap ({len(candidate_map):,})",
    show=True
)

selected_layer = folium.FeatureGroup(
    name=f"3단계: 최종 대표 Hidden Spot ({len(selected_map):,})",
    show=True
)

local_layer = folium.FeatureGroup(
    name=f"4단계: 로컬 상권 위치 ({len(local_map):,})",
    show=True
)

HeatMap(
    H_map[[LAT_COL, LON_COL]].values.tolist(),
    radius=18,
    blur=22,
    min_opacity=0.75,
    max_zoom=13,
    gradient={
        0.10: "#fee5d9",
        0.30: "#fcae91",
        0.50: "#fb6a4a",
        0.70: "#de2d26",
        1.00: "#67000d"
    }
).add_to(h_layer)

h_layer.add_to(m)

candidate_heat_data = []

for _, row in candidate_map.iterrows():
    weight = row.get("covered_h_count", 1)

    if pd.isna(weight):
        weight = 1

    weight = np.log1p(float(weight))

    candidate_heat_data.append([
        row[LAT_COL],
        row[LON_COL],
        weight
    ])

HeatMap(
    candidate_heat_data,
    radius=18,
    blur=22,
    min_opacity=0.65,
    max_zoom=13,
    gradient={
        0.10: "#deebf7",
        0.30: "#9ecae1",
        0.50: "#4292c6",
        0.70: "#08519c",
        1.00: "#08306b"
    }
).add_to(candidate_layer)

candidate_layer.add_to(m)

for _, row in selected_map.iterrows():
    order = int(row["selection_order"])

    popup_text = f"""
    <b>최종 대표 Hidden Spot</b><br><br>
    selection_order: {order}<br>
    space_j_index: {row.get("space_j_index", "NA")}<br>
    j_index: {row.get("j_index", "NA")}<br><br>

    위도: {row.get(LAT_COL, "NA")}<br>
    경도: {row.get(LON_COL, "NA")}<br><br>

    covered_h_count: {row.get("covered_h_count", "NA")}<br>
    avg_x_sim: {row.get("avg_x_sim", "NA")}<br>
    avg_c_sim_scaled: {row.get("avg_c_sim_scaled", "NA")}<br>
    avg_rank: {row.get("avg_rank", "NA")}<br>
    avg_time_diff: {row.get("avg_time_diff", "NA")}<br><br>

    nearest_local_상권명: {row.get("nearest_local_상권명", "NA")}<br>
    nearest_local_자치구: {row.get("nearest_local_자치구", "NA")}<br>
    nearest_local_dist_m: {row.get("nearest_local_dist_m", "NA")} m<br>
    """

    folium.Marker(
        location=[
            row[LAT_COL],
            row[LON_COL]
        ],
        popup=folium.Popup(popup_text, max_width=420),
        tooltip=f"대표 후보 #{order}",
        icon=BeautifyIcon(
            icon_shape="marker",
            number=order,
            border_color="#8b0000",
            background_color="#ff0000",
            text_color="white"
        )
    ).add_to(selected_layer)

selected_layer.add_to(m)

for _, row in local_map.iterrows():
    popup_text = f"""
    <b>로컬 상권</b><br><br>
    기수: {row.get("기수", "NA")}<br>
    선정연도: {row.get("선정연도", "NA")}<br>
    자치구: {row.get("자치구", "NA")}<br>
    상권명: {row.get("상권명", "NA")}<br>
    대표 주소/위치: {row.get("대표 주소/위치", "NA")}<br><br>

    매칭_상권_구분: {row.get("매칭_상권_구분", "NA")}<br>
    매칭_상권_코드: {row.get("매칭_상권_코드", "NA")}<br>
    매칭_상권명: {row.get("매칭_상권명", "NA")}<br>
    매칭_행정동: {row.get("매칭_행정동", "NA")}<br><br>

    위도: {row.get(LAT_COL, "NA")}<br>
    경도: {row.get(LON_COL, "NA")}<br>
    """

    folium.Marker(
        location=[
            row[LAT_COL],
            row[LON_COL]
        ],
        popup=folium.Popup(popup_text, max_width=420),
        tooltip=f"로컬 상권: {row.get('상권명', 'NA')}",
        icon=folium.Icon(
            color="blue",
            icon="info-sign"
        )
    ).add_to(local_layer)

local_layer.add_to(m)

legend_html = f"""
<div style="
position: fixed;
bottom: 40px;
left: 40px;
width: 380px;
height: 240px;
z-index:9999;
font-size:14px;
background-color:white;
border:2px solid grey;
border-radius:10px;
padding: 14px;
box-shadow: 3px 3px 8px rgba(0,0,0,0.3);
">

<b style="font-size:16px;">
Hidden Spot Discovery Map
</b>

<br><br>

<span style="color:#de2d26; font-size:18px;">●</span>
혼잡 지역 H Heatmap

<br>

<span style="color:#08519c; font-size:18px;">●</span>
전체 후보 J Heatmap

<br>

<span style="color:red; font-size:18px;">📍</span>
최종 대표 Hidden Spot

<br>

<span style="color:blue; font-size:18px;">📍</span>
로컬 상권 위치

<br><br>

H count: {len(H_map):,}<br>
J count: {len(candidate_map):,}<br>
Selected Hidden Spot: {len(selected_map):,}<br>
Local Spot: {len(local_map):,}

</div>
"""

m.get_root().html.add_child(folium.Element(legend_html))

folium.LayerControl(collapsed=False).add_to(m)

m.save(str(OUTPUT_PATH))

print("저장 완료:", OUTPUT_PATH)
print("H_map:", H_map.shape)
print("candidate_map:", candidate_map.shape)
print("selected_map:", selected_map.shape)
print("local_map:", local_map.shape)
print("selected unique coords:", selected_map[[LAT_COL, LON_COL]].drop_duplicates().shape[0])

저장 완료: ../RESULT/final_hidden_spot_map_final.html
H_map: (1993504, 6)
candidate_map: (51880, 50)
selected_map: (21, 53)
local_map: (17, 13)
selected unique coords: 21
